read file using pypark and print schema 


In [0]:
df = spark.read.csv("/Volumes/data/orders/files/ecommerce_user_behavior_8000.csv", header=True, inferSchema=True)
df.printSchema()
import pyspark.sql.functions as f

extract sample data 


In [0]:
display(df.limit(10))

Check nulls 

In [0]:
import pyspark.sql.functions as f
null_counts = df.select([
    f.sum(f.when(f.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])
display(null_counts)


In [0]:
invalid_record = df.filter(f.col('user_id').isNull() & f.col('age').isNull() &
                       f.col('gender').isNull() & f.col('device_type').isNull()).count()
display(invalid_record)

dropped rows with null user ID and Purchase is null

In [0]:
print("old count")
display(df.count())
final_df = df.where(f.col('user_id').isNotNull() & f.col('purchase').isNotNull())
print("\nnew count")
display(final_df.count())


fill gender and device type Unknown

In [0]:
final_df = final_df.fillna({'gender':'Unknown','device_type':'Unknown'})
display(final_df.limit(10))

check duplicates and remove them

In [0]:
dup = final_df.groupBy('user_id').count().filter(f.col('count') > 1).count()
print(dup)

fill 0 (dummy value) in remaining columns

In [0]:
final_df = final_df.fillna(0)

check word formation in device type and gender


In [0]:
dis_gender = final_df.select('gender').distinct()
display(dis_gender)
dis_device = final_df.select('device_type').distinct()
display(dis_device)

check correctness of age 

In [0]:
count = final_df.where(f.col('age')<0).count()
display(count)